In [1]:
import sqlite3, os, time

In [2]:
import re

def string_normalize(text: str) -> str:
    """
    Given a text input, 
        - convert to lower case
        - strip leading and trailing white space, replace all other whitespace by underscore, adjacent white spaces would be treated as one white space
    
    Note:
        - assuming input text is text, this function doesn't check and enforce that assumption
    """
    return re.sub(r'\s+', '_', text.strip().lower())


In [4]:
DB_PATH = "test.db"

In [ ]:

# if os.path.exists(DB_PATH):
#     os.remove(DB_PATH)

In [11]:
def attempt(conn, label, sql, params):
    start = time.perf_counter()
    try:
        conn.execute(sql, params)
        print(f"{label:<28} Statement Okay after {time.perf_counter() - start: .2f}s")
        return True 
    except sqlite3.Error as err:
        print(f"{label:<28} Failed with {type(err).__name__} after {time.perf_counter() - start: .2f}s")
        print("| errorname: ", getattr(err, "sqlite_errorname", None),
              "\n| errorcode: ", getattr(err, "sqlite_errorcode", None),
              "\n| errormessage: ", err)
        return False 

def rows(conn):
    result = conn.execute("SELECT * FROM clients ORDER BY id").fetchall()
    return result

In [ ]:
client_table_create = """
    CREATE TABLE clients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        name_standardized TEXT NOT NULL,
        modified_by TEXT NOT NULL,
        CONSTRAINT uq_client_name UNIQUE (name_standardized)
    )
"""

conn = sqlite3.connect(DB_PATH, autocommit=False)
conn.execute(client_table_create)

name = "Farm A"
modified_by = "Alex"

conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", (name.strip(), string_normalize(name), modified_by))
conn.commit()
conn.close()

In [49]:
conn = sqlite3.connect(DB_PATH, autocommit=False, timeout=1)
rows(conn)

[(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex')]

In [58]:
name = "FARM   a"
attempt(conn, "inserting 'FARM a'", "INSERT INTO clients (name, modified_by) VALUES (?, ?)", (name, "Test User"))

inserting 'FARM a'           Failed with IntegrityError after  0.00s
| errorname:  SQLITE_CONSTRAINT_NOTNULL 
| errorcode:  1299 
| errormessage:  NOT NULL constraint failed: clients.name_standardized


False

In [59]:
name = "Farm a"
attempt(conn, "updating 'FARM B' to 'FARM A'", "UPDATE clients SET name=?, name_standardized=?, modified_by=? WHERE id=?", (name, string_normalize(name) , "Test User", 3))

updating 'FARM B' to 'FARM A' Failed with IntegrityError after  0.00s
| errorname:  SQLITE_CONSTRAINT_UNIQUE 
| errorcode:  2067 
| errormessage:  UNIQUE constraint failed: clients.name_standardized


False

In [56]:
# conn.commit()
rows(conn)

[(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex')]

In [60]:
conn.close()

## connection management

In [22]:
def get_connection(db_path: str) -> sqlite3.Connection:
    """
    Establish a connection, enable and verify foreign-key enforcement, return the configured connection
    """
    conn = sqlite3.connect(db_path, autocommit=True, timeout=5)
    conn.execute("PRAGMA foreign_keys = ON")
    fk_check = conn.execute("PRAGMA foreign_keys").fetchone()[0]
    if fk_check != 1:
        conn.close()
        raise RuntimeError("Connection with FK constraint cannot be established.")
    conn.autocommit = False 
    return conn 

In [35]:
from contextlib import contextmanager

@contextmanager
def unit_of_work(db_path: str):
    """
    Actions:
        - acquire database connection
        - execute SQL statements
        - commit on successful completion
        - rollback on exceptions
        - close the connection
    """
    try:
        conn = get_connection(db_path)
    except RuntimeError as err:
        raise RuntimeError("Database connection failed") from err

    try:
        yield conn
        conn.commit()
    except BaseException:
        conn.rollback()
        raise
    finally:
        conn.close()
    

In [24]:
with unit_of_work(DB_PATH) as conn:
    conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm     C", string_normalize("Farm     C"), "Test User 3"))
    

In [26]:
with unit_of_work(DB_PATH) as conn:
    print(rows(conn))

[(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex'), (4, 'Farm     C', 'farm_c', 'Test User 3')]


In [40]:
try:
    with unit_of_work(DB_PATH) as conn:
        conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm E", string_normalize("Farm E"), "Test User 5"))
        conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm E", string_normalize("Farm E"), "Test User 5"))
except BaseException as err:
    print("\n| error type:", type(err).__name__, 
            "\n| error name", getattr(err, "sqlite_errorname", None), 
            "\n| error code", getattr(err, "sqlite_errorcode", None),
            "\n| error message", err)
    


| error type: IntegrityError 
| error name SQLITE_CONSTRAINT_UNIQUE 
| error code 2067 
| error message UNIQUE constraint failed: clients.name_standardized


In [34]:
with unit_of_work(DB_PATH) as conn:
    print(rows(conn))

[(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex'), (4, 'Farm     C', 'farm_c', 'Test User 3'), (5, 'Farm D', 'farm_d', 'Test User 4')]
